In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install librosa soundfile tqdm --quiet

In [1]:
import os
import random
import shutil
import zipfile
from copy import deepcopy

import numpy as np
import pandas as pd
import librosa
import librosa.display
import soundfile as sf
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — required on Kaggle
import matplotlib.pyplot as plt
from tqdm import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('All imports OK')

All imports OK


In [2]:
# ─── PATHS ────────────────────────────────────────────────────────────────────
DATASET_ROOT  = '/kaggle/input/datasets/nasrulhakim86/coughvid-wav/public_dataset'
AUDIO_DIR     = DATASET_ROOT
METADATA_PATH = os.path.join(DATASET_ROOT, '/kaggle/input/datasets/sri09soumya/meta-data/metadata_final_cleaned (1) (3).csv')

OUTPUT_DIR    = '/kaggle/working/coughnet_v2_output'
MEL_NPY_DIR   = os.path.join(OUTPUT_DIR, 'mel_npy')
MEL_PNG_DIR   = os.path.join(OUTPUT_DIR, 'mel_png')
AUG_AUDIO_DIR = os.path.join(OUTPUT_DIR, 'augmented_wavs')

for d in [OUTPUT_DIR, MEL_NPY_DIR, MEL_PNG_DIR, AUG_AUDIO_DIR,
          os.path.join(MEL_NPY_DIR, 'healthy'),
          os.path.join(MEL_NPY_DIR, 'symptomatic'),
          os.path.join(MEL_PNG_DIR, 'healthy'),
          os.path.join(MEL_PNG_DIR, 'symptomatic')]:
    os.makedirs(d, exist_ok=True)

print('Output directories created.')

# ─── AUDIO / MEL SPECTROGRAM PARAMETERS ───────────────────────────────────────
SR          = 22050          # resampled to 22 kHz (changed from 44.1 kHz)
CLIP_DUR    = 3
N_SAMPLES   = SR * CLIP_DUR  # 66150 samples

# Mel spectrogram parameters — produces 128×128 images
N_FFT       = 512
HOP_LENGTH  = 512
N_MELS      = 128

# Cough detection threshold
COUGH_THRESH = 0.5

# ─── AUGMENTATION TARGETS ─────────────────────────────────────────────────────
TARGET = 6000

print(f'Sample rate       : {SR} Hz')
print(f'Augmentation target per class: {TARGET}')
mel_frames = (N_SAMPLES // HOP_LENGTH) + 1
print(f'Expected Mel spectrogram shape: {N_MELS} mels × ~{mel_frames} frames  '
      f'(will be resized to 128×128)')

Output directories created.
Sample rate       : 22050 Hz
Augmentation target per class: 6000
Expected Mel spectrogram shape: 128 mels × ~130 frames  (will be resized to 128×128)


In [3]:
df = pd.read_csv(METADATA_PATH)
print(f'Raw metadata rows: {len(df)}')
print(df['status'].value_counts())
print(df.head(3))

Raw metadata rows: 16224
status
healthy        12479
symptomatic     2590
COVID-19        1155
Name: count, dtype: int64
                           datetime  cough_detected   age  gender  \
0  2020-06-21T23:34:58.482745+00:00          0.9887  28.0  female   
1  2020-08-21T12:10:03.711574+00:00          0.0113  25.0    male   
2  2020-05-11T13:01:04.648992+00:00          0.1100  43.0  female   

   respiratory_condition  fever_muscle_pain   status  \
0                  False              False  healthy   
1                  False              False  healthy   
2                  False              False  healthy   

                                   uuid  
0  bfe39f99-20e1-471d-a6c0-e82effbba33a  
1  8b48c362-a7f5-4db7-8a63-d116f7458b5f  
2  93a49de4-abfb-4d83-9504-4cdc6028d98c  


In [ ]:
# ── 2a. Merge COVID-19 into symptomatic (paper's decision: no RT-PCR reports)
df['label'] = df['status'].apply(
    lambda s: 'symptomatic' if s in ('symptomatic', 'COVID-19') else 'healthy'
)

# ── 2b. Drop silent files
df = df[df['cough_detected'] >= COUGH_THRESH].copy()
print(f'After cough_detected >= {COUGH_THRESH}: {len(df)} rows')
print(df['label'].value_counts())

# ── 2c. Drop rows missing required medical fields
required_cols = ['age', 'gender', 'respiratory_condition', 'fever_muscle_pain']
df = df.dropna(subset=required_cols).copy()
print(f'After dropping rows with missing medical info: {len(df)} rows')
print(df['label'].value_counts())

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Augmentation Helpers — symptomatic/COVID-19 samples ONLY
# ──────────────────────────────────────────────────────────────────────────────

def augment_pitch_shift(y: np.ndarray, sr: int = SR,
                         n_steps: float = -4.0) -> np.ndarray:
    """
    Pitch-shift the waveform by a fixed n_steps = -4 semitones (per paper spec).
    Applied at waveform level before Mel extraction.
    """
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps).astype(np.float32)


def apply_spec_augment(mel_db: np.ndarray,
                        F: int = 30,
                        T: int = 30) -> np.ndarray:
    """
    SpecAugment: apply one frequency mask (width F) and one time mask (width T)
    to a Mel spectrogram (shape: n_mels × time_frames), in-place on a copy.
    F=30, T=30 per paper specification.
    """
    mel_aug = mel_db.copy()
    n_mels, n_frames = mel_aug.shape

    # Frequency masking — mask F consecutive mel bins
    f0 = np.random.randint(0, max(1, n_mels - F))
    mel_aug[f0:f0 + F, :] = mel_aug.min()

    # Time masking — mask T consecutive time frames
    t0 = np.random.randint(0, max(1, n_frames - T))
    mel_aug[:, t0:t0 + T] = mel_aug.min()

    return mel_aug


# Only two augmentation types — applied exclusively to symptomatic samples
AUG_FUNCTIONS = [
    ('pitch_shift',  augment_pitch_shift),   # waveform-level
    ('spec_augment', None),                  # spectrogram-level — handled separately
]

print(f'Defined {len(AUG_FUNCTIONS)} augmentation functions: '
      + ', '.join(n for n, _ in AUG_FUNCTIONS))

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Audio Loading Helper
# ──────────────────────────────────────────────────────────────────────────────

def load_and_pad_audio(uuid: str, audio_dir: str,
                        sr: int = SR, duration: int = CLIP_DUR):
    """
    Load a .wav file by UUID, resample to `sr` (22050 Hz), and trim/pad
    to `duration` seconds. Returns (signal, True) on success, (None, False) on failure.
    """
    path = os.path.join(audio_dir, f'{uuid}.wav')
    if not os.path.exists(path):
        return None, False
    try:
        y, orig_sr = librosa.load(path, sr=sr, mono=True)  # resamples to 22 kHz
        target_len = sr * duration
        y = y[:target_len] if len(y) >= target_len else np.pad(y, (0, target_len - len(y)))
        return y.astype(np.float32), True
    except Exception:
        return None, False

print('Audio loader defined.')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Build Balanced DataFrames via Augmentation (symptomatic only)
# ──────────────────────────────────────────────────────────────────────────────

symp_df    = df[df['label'] == 'symptomatic'].copy()
healthy_df = df[df['label'] == 'healthy'].copy()

print(f'Available symptomatic : {len(symp_df)}')
print(f'Available healthy     : {len(healthy_df)}')
print(f'Target per class      : {TARGET}')

# Sample healthy DOWN to target (no augmentation on healthy class)
if len(healthy_df) > TARGET:
    healthy_df = healthy_df.sample(n=TARGET, random_state=SEED)
    print(f'Healthy sampled to {TARGET}')

n_symp_orig   = len(symp_df)
n_symp_needed = TARGET - n_symp_orig
print(f'\nSymptomatic originals : {n_symp_orig}')
print(f'Symptomatic to augment: {n_symp_needed}')

# Build augmentation metadata rows — only pitch_shift and spec_augment
aug_rows = []
aug_idx  = 0
symp_pool = symp_df.to_dict('records')
pool_len  = len(symp_pool)

# Alternate between pitch_shift and spec_augment
aug_cycle = ['pitch_shift', 'spec_augment']

for i in range(n_symp_needed):
    src_row  = symp_pool[i % pool_len]
    aug_name = aug_cycle[i % len(aug_cycle)]

    new_row = deepcopy(src_row)
    new_row['uuid']         = f'aug_{aug_name}_{aug_idx:05d}_{src_row["uuid"]}'
    new_row['augmentation'] = aug_name
    new_row['source_uuid']  = src_row['uuid']
    aug_rows.append(new_row)
    aug_idx += 1

aug_df = pd.DataFrame(aug_rows)

symp_df['augmentation']    = 'none'
symp_df['source_uuid']     = symp_df['uuid']
healthy_df['augmentation'] = 'none'
healthy_df['source_uuid']  = healthy_df['uuid']

df_balanced = pd.concat([symp_df, aug_df, healthy_df])\
                .sample(frac=1, random_state=SEED)\
                .reset_index(drop=True)

print(f'\nFinal balanced dataset: {len(df_balanced)} rows')
print(df_balanced['label'].value_counts())
print('\nAugmentation technique distribution:')
print(df_balanced[df_balanced['augmentation'] != 'none']['augmentation'].value_counts())

In [ ]:
def extract_mel_spectrogram(y: np.ndarray,
                              sr: int   = SR,
                              n_fft: int = N_FFT,
                              hop: int   = HOP_LENGTH,
                              n_mels: int = N_MELS,
                              img_size: int = 128) -> np.ndarray:
    """
    Compute Mel spectrogram and return a 128×128 image normalised to [0, 1].

    Pipeline:
      1. librosa.feature.melspectrogram  (n_fft=512, hop_length=512, n_mels=128)
      2. Convert power to dB scale
      3. Resize to img_size × img_size  (128×128)
      4. Normalise to [0, 1]
    """
    # Step 1 — Mel spectrogram  (shape: n_mels × time_frames)
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr,
        n_fft=n_fft,
        hop_length=hop,
        n_mels=n_mels,
        center=False
    )

    # Step 2 — Convert to dB
    mel_db = librosa.power_to_db(mel, ref=np.max)  # shape: (128, T)

    # Step 3 — Resize to 128×128 using bilinear interpolation via PIL
    from PIL import Image
    img = Image.fromarray(mel_db).resize((img_size, img_size), Image.BILINEAR)
    mel_img = np.array(img, dtype=np.float32)  # shape: (128, 128)

    # Step 4 — Normalise to [0, 1]
    min_val, max_val = mel_img.min(), mel_img.max()
    if max_val - min_val > 1e-6:
        mel_img = (mel_img - min_val) / (max_val - min_val)
    else:
        mel_img = np.zeros_like(mel_img)

    return mel_img  # shape: (128, 128), dtype float32, values in [0, 1]


# Sanity check
dummy = np.random.randn(N_SAMPLES).astype(np.float32)
test_mel = extract_mel_spectrogram(dummy)
print(f'Test Mel spectrogram shape : {test_mel.shape}   (expected: (128, 128))')
print(f'Value range                : [{test_mel.min():.4f}, {test_mel.max():.4f}]  (expected: [0, 1])')

In [ ]:
def save_mel_png(mel_img: np.ndarray, out_path: str):
    """Save the 128×128 normalised Mel spectrogram image as a PNG."""
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(mel_img, origin='lower', aspect='auto', cmap='viridis',
              vmin=0, vmax=1)
    ax.set_title('Mel Spectrogram (normalised)', fontsize=9)
    ax.set_xlabel('Time (frames)')
    ax.set_ylabel('Mel bins')
    plt.tight_layout()
    plt.savefig(out_path, dpi=100, bbox_inches='tight')
    plt.close(fig)

print('PNG saver defined.')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Main loop — with inline augmentation
#   pitch_shift  → waveform-level (before Mel extraction)
#   spec_augment → spectrogram-level (after Mel extraction, before normalisation)
# ──────────────────────────────────────────────────────────────────────────────

records     = []
skipped     = 0
shape_check = set()

for _, row in tqdm(df_balanced.iterrows(), total=len(df_balanced), desc='Extracting Mel spectrograms'):
    uuid     = row['uuid']
    label    = row['label']
    aug_type = row.get('augmentation', 'none')
    src_uuid = row.get('source_uuid', uuid)

    # ── Load source audio (resampled to 22 kHz inside load_and_pad_audio)
    y, ok = load_and_pad_audio(src_uuid, AUDIO_DIR)
    if not ok:
        skipped += 1
        continue

    # ── Waveform-level augmentation: pitch shift (symptomatic only, n_steps=-4)
    if aug_type == 'pitch_shift':
        try:
            y = augment_pitch_shift(y, sr=SR, n_steps=-4)
        except Exception:
            pass  # fall back to original waveform

    # ── Extract Mel spectrogram  →  shape (128, T)
    mel = librosa.feature.melspectrogram(
        y=y, sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        center=False
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)  # (128, T)

    # ── Spectrogram-level augmentation: SpecAugment (symptomatic only, F=30, T=30)
    if aug_type == 'spec_augment':
        mel_db = apply_spec_augment(mel_db, F=30, T=30)

    # ── Resize to 128×128 and normalise to [0, 1]
    from PIL import Image
    img = Image.fromarray(mel_db).resize((128, 128), Image.BILINEAR)
    mel_img = np.array(img, dtype=np.float32)
    min_val, max_val = mel_img.min(), mel_img.max()
    if max_val - min_val > 1e-6:
        mel_img = (mel_img - min_val) / (max_val - min_val)
    else:
        mel_img = np.zeros_like(mel_img)

    shape_check.add(mel_img.shape)

    # ── Save .npy  (shape: 128×128, float32, [0,1])
    npy_path = os.path.join(MEL_NPY_DIR, label, f'{uuid}.npy')
    np.save(npy_path, mel_img)

    # ── Save .png
    png_path = os.path.join(MEL_PNG_DIR, label, f'{uuid}.png')
    save_mel_png(mel_img, png_path)

    # ── Record metadata
    records.append({
        'uuid':                  uuid,
        'source_uuid':           src_uuid,
        'augmentation':          aug_type,
        'label':                 label,
        'age':                   row['age'],
        'gender':                row['gender'],
        'respiratory_condition': row['respiratory_condition'],
        'fever_muscle_pain':     row['fever_muscle_pain'],
        'cough_detected':        row['cough_detected'],
        'mel_npy_path':          npy_path,
        'mel_png_path':          png_path,
        'mel_shape':             str(mel_img.shape),
    })

print(f'\nDone! Processed: {len(records)}  |  Skipped (file not found): {skipped}')
print(f'Unique Mel spectrogram shapes encountered: {shape_check}')

In [ ]:
df_out = pd.DataFrame(records)
print(df_out['label'].value_counts())
print('\nAugmentation breakdown:')
print(df_out.groupby(['label', 'augmentation']).size())

meta_csv_path = os.path.join(OUTPUT_DIR, 'processed_metadata.csv')
df_out.to_csv(meta_csv_path, index=False)
print(f'\nSaved processed metadata → {meta_csv_path}')
df_out.head(3)

In [ ]:
# Compare: original | pitch_shift | spec_augment — symptomatic samples only
aug_types_available = df_out[df_out['augmentation'] != 'none']['augmentation'].unique().tolist()
n_plots = 1 + len(aug_types_available)
fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4))

# Original symptomatic
orig_row  = df_out[(df_out['label'] == 'symptomatic') & (df_out['augmentation'] == 'none')].iloc[0]
mel_data  = np.load(orig_row['mel_npy_path'])
axes[0].imshow(mel_data, origin='lower', aspect='auto', cmap='viridis', vmin=0, vmax=1)
axes[0].set_title('Symptomatic — Original')
axes[0].set_xlabel('Time (frames)')
axes[0].set_ylabel('Mel bins')

# One per augmentation type
for ax, aug in zip(axes[1:], aug_types_available):
    aug_row  = df_out[df_out['augmentation'] == aug].iloc[0]
    mel_data = np.load(aug_row['mel_npy_path'])
    ax.imshow(mel_data, origin='lower', aspect='auto', cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f'Symptomatic — {aug}')
    ax.set_xlabel('Time (frames)')

plt.suptitle('Original vs Augmented Mel Spectrograms — Symptomatic (COVID-19) Class', fontsize=13)
plt.tight_layout()
preview_path = os.path.join(OUTPUT_DIR, 'augmentation_comparison.png')
plt.savefig(preview_path, dpi=120)
plt.show()
print(f'Preview saved → {preview_path}')

In [ ]:
# Step 9 — Zip
ZIP_PATH = '/kaggle/working/coughnet_v2_mel_dataset.zip'

print('Zipping output directory...')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            arcname   = os.path.relpath(full_path, '/kaggle/working')
            zf.write(full_path, arcname)

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f'Zip created: {ZIP_PATH}  ({size_mb:.1f} MB)')

In [ ]:
# ── Save ONLY augmented mel spectrogram PNGs as a zip ─────────────────────────
import zipfile, os

AUG_PNG_ZIP = '/kaggle/working/augmented_mel_specs.zip'

print('Zipping augmented mel spec PNGs...')
with zipfile.ZipFile(AUG_PNG_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for label in ['healthy', 'symptomatic']:
        png_label_dir = os.path.join(MEL_PNG_DIR, label)
        for fname in os.listdir(png_label_dir):
            if fname.endswith('.png'):
                full_path = os.path.join(png_label_dir, fname)
                # Include the label subfolder in the zip path
                arcname = os.path.join(label, fname)
                zf.write(full_path, arcname)

size_mb = os.path.getsize(AUG_PNG_ZIP) / 1e6
print(f'Done! Zip saved to: {AUG_PNG_ZIP}  ({size_mb:.1f} MB)')
print('Go to Kaggle sidebar → Output tab → Download augmented_mel_specs.zip')

In [ ]:
import shutil
total, used, free = shutil.disk_usage('/kaggle/working')
print(f'Used : {used / 1e9:.2f} GB')
print(f'Free : {free / 1e9:.2f} GB')

# Also check if the zip actually exists
import os
zip_path = '/kaggle/working/augmented_mel_specs.zip'
if os.path.exists(zip_path):
    print(f'Zip exists: {os.path.getsize(zip_path)/1e6:.1f} MB')
else:
    print('Zip NOT found — it was never created or was deleted')

In [4]:
import numpy as np
import pandas as pd
import os
import random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau,
                                        ModelCheckpoint, CSVLogger)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')


2026-05-02 07:45:35.409936: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777707935.603153     275 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777707935.659469     275 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777707936.144328     275 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777707936.144372     275 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777707936.144374     275 computation_placer.cc:177] computation placer alr

In [5]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version : {tf.__version__}")
print(f"GPUs available     : {tf.config.list_physical_devices('GPU')}")

TensorFlow version : 2.19.0
GPUs available     : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [6]:
# df_out was saved in section 8; reload if needed
meta_csv_path = os.path.join(OUTPUT_DIR, 'processed_metadata.csv')
df_out = pd.read_csv(meta_csv_path)

# Binary label: COVID/symptomatic → 1,  healthy → 0
df_out['binary_label'] = (df_out['label'] == 'symptomatic').astype(int)

train_df, temp_df = train_test_split(
    df_out, test_size=0.20, stratify=df_out['binary_label'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['binary_label'], random_state=SEED
)

print(f"Train : {len(train_df)}  "
      f"(COVID={train_df['binary_label'].sum()}, "
      f"Healthy={len(train_df)-train_df['binary_label'].sum()})")
print(f"Val   : {len(val_df)}  "
      f"(COVID={val_df['binary_label'].sum()}, "
      f"Healthy={len(val_df)-val_df['binary_label'].sum()})")
print(f"Test  : {len(test_df)}  "
      f"(COVID={test_df['binary_label'].sum()}, "
      f"Healthy={len(test_df)-test_df['binary_label'].sum()})")

Train : 9600  (COVID=4800, Healthy=4800)
Val   : 1200  (COVID=600, Healthy=600)
Test  : 1200  (COVID=600, Healthy=600)


In [9]:
IMG_SIZE   = 128
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE


def load_npy(path: str, label: int):
    """Load a single .npy mel spectrogram and expand to (128,128,1)."""
    mel = np.load(path).astype(np.float32)        # (128, 128)
    mel = mel[..., np.newaxis]                    # (128, 128, 1)
    return mel, label

def make_dataset(df: pd.DataFrame, shuffle: bool = False) -> tf.data.Dataset:
    paths  = df['mel_npy_path'].values
    labels = df['binary_label'].values.astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)

    # ── FIX: wrap load_npy so it always returns float32 for both outputs ──
    def _load(path_bytes, label):
        mel = np.load(path_bytes.decode()).astype(np.float32)
        mel = mel[..., np.newaxis]                        # (128, 128, 1)
        return mel, np.float32(label)                     # ← explicit cast here

    ds = ds.map(
        lambda p, y: tf.numpy_function(
            func=_load,
            inp=[p, y],
            Tout=[tf.float32, tf.float32]                 # both float32
        ),
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.map(
        lambda x, y: (tf.ensure_shape(x, [IMG_SIZE, IMG_SIZE, 1]),
                      tf.ensure_shape(y, [])),
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


train_ds = make_dataset(train_df, shuffle=True)
val_ds   = make_dataset(val_df,   shuffle=False)
test_ds  = make_dataset(test_df,  shuffle=False)

# Sanity check
for x_batch, y_batch in train_ds.take(1):
    print(f"x dtype: {x_batch.dtype}  shape: {x_batch.shape}")
    print(f"y dtype: {y_batch.dtype}  shape: {y_batch.shape}")


x dtype: <dtype: 'float32'>  shape: (32, 128, 128, 1)
y dtype: <dtype: 'float32'>  shape: (32,)


In [10]:
CLASS_WEIGHTS = {1: 7.01, 0: 0.54}    # as specified in the paper
print(f"Class weights → COVID (1): {CLASS_WEIGHTS[1]}  |  Healthy (0): {CLASS_WEIGHTS[0]}")

Class weights → COVID (1): 7.01  |  Healthy (0): 0.54


Model compiled — Optimizer: Adam  LR: 0.0001  Loss: binary_crossentropy


In [14]:
def conv_block(x, filters: int, dropout_rate: float):
    """Two Conv2D (ReLU) → BatchNorm → MaxPool → Dropout."""
    x = layers.Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(dropout_rate)(x)
    return x


def build_cnn(input_shape=(128, 128, 1)) -> keras.Model:
    inputs = keras.Input(shape=input_shape)

    # Block 1
    x = conv_block(inputs, filters=32,  dropout_rate=0.25)
    # Block 2
    x = conv_block(x,      filters=64,  dropout_rate=0.25)
    # Block 3
    x = conv_block(x,      filters=128, dropout_rate=0.30)
    # Block 4
    x = conv_block(x,      filters=256, dropout_rate=0.30)

    # Global average pooling (replaces Flatten + Dense head — reduces overfitting)
    x = layers.GlobalAveragePooling2D()(x)

    # Binary sigmoid output
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs, outputs, name='CoughNet_CNN')
    return model


model = build_cnn()
model.summary()

total_params = model.count_params()
print(f"\nTotal trainable parameters: {total_params:,}")


# ══════════════════════════════════════════════════════════════════════════════
# 10-E  Compile
#   Optimizer  : Adam  (η = 1e-4)
#   Loss       : binary cross-entropy
#   Metrics    : accuracy, AUC-ROC, Precision, Recall
# ══════════════════════════════════════════════════════════════════════════════

LR = 1e-4

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.AUC(name='auc_roc', curve='ROC'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ]
)

print(f"Model compiled — Optimizer: Adam  LR: {LR}  Loss: binary_crossentropy")



Model: "CoughNet_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 128, 128, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 128, 128, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_22 (Conv2D)              │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 16, 16, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 1,173,857 (4.48 MB)

 Trainable params: 1,172,897 (4.47 MB)

 Non-trainable params: 960 (3.75 KB)


Total trainable parameters: 1,173,857
Model compiled — Optimizer: Adam  LR: 0.0001  Loss: binary_crossentropy


In [15]:
CKPT_PATH = os.path.join(OUTPUT_DIR, 'best_cnn_model.keras')
LOG_PATH  = os.path.join(OUTPUT_DIR, 'training_log.csv')

callbacks = [
    # Stop early if val_auc_roc doesn't improve for 5 epochs
    EarlyStopping(
        monitor='val_auc_roc', mode='max',
        patience=5, restore_best_weights=True, verbose=1
    ),
    # Halve LR after 3 stagnant epochs
    ReduceLROnPlateau(
        monitor='val_auc_roc', mode='max',
        factor=0.5, patience=3, min_lr=1e-7, verbose=1
    ),
    # Save the best checkpoint
    ModelCheckpoint(
        filepath=CKPT_PATH, monitor='val_auc_roc',
        mode='max', save_best_only=True, verbose=1
    ),
    # Epoch-by-epoch CSV log
    CSVLogger(LOG_PATH)
]

print("Callbacks configured.")



Callbacks configured.


In [19]:
# 10-G  Training  (20 epochs, batch size 32, class weights as specified)
# ══════════════════════════════════════════════════════════════════════════════

EPOCHS = 20

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print(f"\nTraining complete. Best model saved → {CKPT_PATH}")

Epoch 1/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.6890 - auc_roc: 0.8012 - loss: 0.5622 - precision: 0.7097 - recall: 0.7179
Epoch 1: val_auc_roc improved from 0.78513 to 0.78710, saving model to /kaggle/working/coughnet_v2_output/best_cnn_model.keras
300/300 ━━━━━━━━━━━━━━━━━━━━ 26s 59ms/step - accuracy: 0.6892 - auc_roc: 0.8012 - loss: 0.5620 - precision: 0.7100 - recall: 0.7176 - val_accuracy: 0.6758 - val_auc_roc: 0.7871 - val_loss: 0.5231 - val_precision: 0.6549 - val_recall: 0.7433 - learning_rate: 5.0000e-05
Epoch 2/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.7641 - auc_roc: 0.8300 - loss: 0.4575 - precision: 0.8874 - recall: 0.6097
Epoch 2: val_auc_roc did not improve from 0.78710
300/300 ━━━━━━━━━━━━━━━━━━━━ 18s 60ms/step - accuracy: 0.7641 - auc_roc: 0.8300 - loss: 0.4575 - precision: 0.8874 - recall: 0.6097 - val_accuracy: 0.6617 - val_auc_roc: 0.7745 - val_loss: 0.5540 - val_precision: 0.6431 - val_recall: 0.7267 - learning_rate: 5.0000e-05
E

In [20]:
# Load best checkpoint before evaluating
model = keras.models.load_model(CKPT_PATH)
print(f"Loaded best model from {CKPT_PATH}")

# Get raw predictions
y_true, y_prob = [], []
for x_batch, y_batch in test_ds:
    probs = model.predict(x_batch, verbose=0).flatten()
    y_prob.extend(probs.tolist())
    y_true.extend(y_batch.numpy().tolist())

y_true = np.array(y_true)
y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

# ── Keras built-in metrics on test set
test_results = model.evaluate(test_ds, verbose=1)
metric_names  = ['loss', 'accuracy', 'auc_roc', 'precision', 'recall']
print("\n── Keras Test Metrics ──────────────────────────────────")
for name, val in zip(metric_names, test_results):
    print(f"  {name:<15}: {val:.4f}")

# ── Scikit-learn detailed report
print("\n── Classification Report ───────────────────────────────")
print(classification_report(y_true, y_pred,
                             target_names=['Healthy (0)', 'COVID (1)'],
                             digits=4))

# ── Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)          # Recall for COVID class
specificity = tn / (tn + fp)
f1_covid    = 2 * tp / (2 * tp + fp + fn)
auc_roc     = roc_auc_score(y_true, y_prob)

print("── Key Clinical Metrics ────────────────────────────────")
print(f"  Sensitivity (COVID Recall) : {sensitivity:.4f}")
print(f"  Specificity                : {specificity:.4f}")
print(f"  F1 Score (COVID)           : {f1_covid:.4f}")
print(f"  AUC-ROC                    : {auc_roc:.4f}")


Loaded best model from /kaggle/working/coughnet_v2_output/best_cnn_model.keras
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.7276 - auc_roc: 0.7643 - loss: 0.5050 - precision: 0.8608 - recall: 0.5723

── Keras Test Metrics ──────────────────────────────────
  loss           : 0.5042
  accuracy       : 0.7375
  auc_roc        : 0.7654
  precision      : 0.8571
  recall         : 0.5700

── Classification Report ───────────────────────────────
              precision    recall  f1-score   support

 Healthy (0)     0.6779    0.9050    0.7752       600
   COVID (1)     0.8571    0.5700    0.6847       600

    accuracy                         0.7375      1200
   macro avg     0.7675    0.7375    0.7299      1200
weighted avg     0.7675    0.7375    0.7299      1200

── Key Clinical Metrics ────────────────────────────────
  Sensitivity (COVID Recall) : 0.5700
  Specificity                : 0.9050
  F1 Score (COVID)           : 0.6847
  AUC-ROC                    : 0.7653


In [18]:
print(train_df['binary_label'].value_counts())
print(val_df['binary_label'].value_counts())
print(test_df['binary_label'].value_counts())

binary_label
0    4800
1    4800
Name: count, dtype: int64
binary_label
1    600
0    600
Name: count, dtype: int64
binary_label
1    600
0    600
Name: count, dtype: int64
